# PYNQ-Z2 AD9226 Capture Full Test

测试顺序：

1. 加载 `base_add.bit`
2. 检查 IP
3. 测 LED / AXI-Lite
4. `capture_mode = 0`：HLS fake → DDR → PS
5. `capture_mode = 2`：RTL fake stream → FIFO → HLS → DDR → PS
6. `capture_mode = 1`：真实 AD9226 → FIFO → HLS → DDR → PS

先不要一上来测 62.5MHz。建议先从低速开始。

In [ ]:
from pynq import Overlay, allocate
import numpy as np
import matplotlib.pyplot as plt
from time import time, sleep

overlay = Overlay("base_add.bit")

print("IP list:")
for k in overlay.ip_dict.keys():
    print("  ", k)

# 按当前工程报告，通常是这两个名字。
# 如果你的 ip_dict 打印出来名字不同，请在这里改。
writer = overlay.base_add_0
ctrl = overlay.adc_capture_0

print("\nwriter register_map:")
print(writer.register_map)

print("\nadc ctrl register_map:")
print(ctrl.register_map)

## 1. 裸地址定义和公共函数

注意：`ip.write(offset, value)` 使用的是 IP 内部 offset，不要加 Vivado base address。

In [ ]:
# ===== adc_capture_0 registers =====
CTRL          = 0x00
STATUS        = 0x04
SAMPLE_COUNT  = 0x08
ADC_HALF      = 0x0C
SAMPLE_DELAY  = 0x10
DECIMATION    = 0x14
CHANNEL_MASK  = 0x18
CAPTURE_MODE  = 0x1C
TRIGGER_MODE  = 0x20
PRE_DELAY     = 0x24
BUFFER_SELECT = 0x28
LATEST_A      = 0x2C
LATEST_B      = 0x30
SAMPLE_CNTR   = 0x34
FIFO_LEVEL    = 0x38
ERROR_FLAGS   = 0x3C
LED_CTRL      = 0x40
VERSION       = 0x44
SAVED_COUNTER = 0x48
LAST_SAMPLE   = 0x4C
DEBUG_STATE   = 0x50

# ===== HLS writer registers from HLS report =====
WRITER_AP_CTRL      = 0x00
WRITER_BUFFER       = 0x10
WRITER_COUNT        = 0x18
WRITER_CAPTURE_MODE = 0x20

# ===== constants =====
MAX_SAMPLE_N = 65536
BUFFER_WORDS = 2 * MAX_SAMPLE_N

def read_status():
    status = ctrl.read(STATUS)
    err = ctrl.read(ERROR_FLAGS)
    fifo = ctrl.read(FIFO_LEVEL)
    saved = ctrl.read(SAVED_COUNTER)
    last = ctrl.read(LAST_SAMPLE)
    dbg = ctrl.read(DEBUG_STATE)
    writer_ctrl = writer.read(WRITER_AP_CTRL)

    print("STATUS       = 0x%08X" % status)
    print("ERROR_FLAGS  = 0x%08X" % err)
    print("FIFO_LEVEL   =", fifo)
    print("SAVED_CNTR   =", saved)
    print("LAST_SAMPLE  = 0x%08X" % last)
    print("DEBUG_STATE  =", dbg)
    print("WRITER_CTRL  = 0x%08X" % writer_ctrl)
    print("LATEST_A     =", ctrl.read(LATEST_A))
    print("LATEST_B     =", ctrl.read(LATEST_B))
    print("SAMPLE_CNTR  =", ctrl.read(SAMPLE_CNTR))
    print("VERSION      = 0x%08X" % ctrl.read(VERSION))

def configure_ctrl(sample_count=1024,
                   adc_half_period=6,
                   sample_delay=1,
                   decimation=1,
                   channel_mask=0b11,
                   capture_mode=0,
                   trigger_mode=0,
                   pre_delay=0,
                   buffer_select=0,
                   led_ps_override=0,
                   led_value=0):

    sample_count = min(max(int(sample_count), 1), MAX_SAMPLE_N)
    adc_half_period = max(int(adc_half_period), 1)
    decimation = max(int(decimation), 1)

    # clear pulse, then release
    ctrl.write(CTRL, 0x04)
    ctrl.write(CTRL, 0x00)

    ctrl.write(SAMPLE_COUNT, sample_count)
    ctrl.write(ADC_HALF, adc_half_period)
    ctrl.write(SAMPLE_DELAY, sample_delay)
    ctrl.write(DECIMATION, decimation)
    ctrl.write(CHANNEL_MASK, channel_mask)
    ctrl.write(CAPTURE_MODE, capture_mode)
    ctrl.write(TRIGGER_MODE, trigger_mode)
    ctrl.write(PRE_DELAY, pre_delay)
    ctrl.write(BUFFER_SELECT, buffer_select)
    ctrl.write(LED_CTRL, (led_ps_override << 8) | (led_value & 0xF))

    # writer 也必须同步写 capture_mode
    writer.write(WRITER_CAPTURE_MODE, capture_mode)

    actual_adc_fs = 125_000_000 / (2 * adc_half_period)
    actual_save_fs = actual_adc_fs / decimation

    print("sample_count    =", sample_count)
    print("capture_mode    =", capture_mode)
    print("adc_half_period =", adc_half_period)
    print("actual_adc_fs   =", actual_adc_fs)
    print("decimation      =", decimation)
    print("actual_save_fs  =", actual_save_fs)

    return sample_count, actual_adc_fs, actual_save_fs

def run_capture(sample_count, capture_mode, timeout_s=5.0):
    buf = allocate(shape=(BUFFER_WORDS,), dtype=np.int32)
    buf[:] = -12345
    buf.flush()

    writer.write(WRITER_BUFFER, int(buf.physical_address) & 0xFFFFFFFF)
    writer.write(WRITER_COUNT, sample_count)
    writer.write(WRITER_CAPTURE_MODE, capture_mode)

    if capture_mode == 0:
        # HLS fake，不需要启动 capture_core
        writer.write(WRITER_AP_CTRL, 0x01)
    else:
        # stream 模式：writer 先启动，再 start capture
        ctrl.write(CTRL, 0x01)      # enable
        writer.write(WRITER_AP_CTRL, 0x01)
        ctrl.write(CTRL, 0x03)      # enable + start pulse
        ctrl.write(CTRL, 0x01)      # release start

    t0 = time()
    while True:
        ctrl_status = ctrl.read(STATUS)
        writer_ctrl = writer.read(WRITER_AP_CTRL)

        writer_done = (writer_ctrl & 0x2) != 0
        ctrl_done = (ctrl_status & (1 << 1)) != 0

        if capture_mode == 0:
            done = writer_done
        else:
            done = writer_done and ctrl_done

        if done:
            break

        if time() - t0 > timeout_s:
            print("TIMEOUT DEBUG:")
            read_status()
            raise TimeoutError("capture timeout")

    buf.invalidate()

    ch0 = np.array(buf[0:sample_count], dtype=np.int32)
    ch1 = np.array(buf[MAX_SAMPLE_N:MAX_SAMPLE_N + sample_count], dtype=np.int32)

    return buf, ch0, ch1

def plot_capture(ch0, ch1, fs=None, title="capture", max_points=None):
    if max_points is not None:
        ch0 = ch0[:max_points]
        ch1 = ch1[:max_points]

    n = len(ch0)
    if fs is None:
        x = np.arange(n)
        xlabel = "sample index"
    else:
        x = np.arange(n) / fs
        xlabel = "time / s"

    plt.figure(figsize=(12, 4))
    plt.plot(x, ch0, label="CH0")
    plt.plot(x, ch1, label="CH1")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("raw code")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

    print("CH0 first 8:", ch0[:8])
    print("CH1 first 8:", ch1[:8])
    print("CH0 mean/vpp:", float(np.mean(ch0)), int(np.max(ch0) - np.min(ch0)))
    print("CH1 mean/vpp:", float(np.mean(ch1)), int(np.max(ch1) - np.min(ch1)))

## 2. LED / AXI-Lite 基础测试

这一格不需要接 AD9226。能亮说明 AXI-Lite 和 overlay 基本正常。

In [ ]:
print("VERSION:", hex(ctrl.read(VERSION)))

# PS 手动控制 LED
for v in [0x1, 0x2, 0x4, 0x8, 0xF, 0x0]:
    ctrl.write(LED_CTRL, (1 << 8) | v)
    print("LED =", hex(v))
    sleep(0.3)

# 恢复自动 LED
ctrl.write(LED_CTRL, 0x000)
read_status()

## 3. Mode 0：HLS fake → DDR → PS

验证 HLS writer、DDR、PYNQ allocate、cache flush/invalidate。  
不经过 RTL capture/FIFO，不需要接 AD9226。

In [ ]:
sample_count, actual_adc_fs, actual_save_fs = configure_ctrl(
    sample_count=1024,
    adc_half_period=6,
    sample_delay=1,
    decimation=1,
    channel_mask=0b11,
    capture_mode=0
)

buf, ch0, ch1 = run_capture(sample_count, capture_mode=0)
read_status()
plot_capture(ch0, ch1, title="mode 0: HLS fake -> DDR")

## 4. Mode 0 大 buffer 快速检查

确认 16384 点时 DDR buffer 和 writer 也正常。只画前 1024 点。

In [ ]:
sample_count, actual_adc_fs, actual_save_fs = configure_ctrl(
    sample_count=16384,
    adc_half_period=6,
    capture_mode=0
)

buf, ch0, ch1 = run_capture(sample_count, capture_mode=0)
read_status()
plot_capture(ch0, ch1, title="mode 0: first 1024 of 16384", max_points=1024)

## 5. Mode 2：RTL fake stream → FIFO → HLS → DDR → PS

这是关键中间测试。  
不需要接 AD9226，但会经过 RTL capture_core、FIFO、AXI-Stream、HLS stream 输入。

In [ ]:
sample_count, actual_adc_fs, actual_save_fs = configure_ctrl(
    sample_count=1024,
    adc_half_period=6,
    sample_delay=1,
    decimation=1,
    channel_mask=0b11,
    capture_mode=2
)

buf, ch0, ch1 = run_capture(sample_count, capture_mode=2)
read_status()
plot_capture(ch0, ch1, fs=actual_save_fs, title="mode 2: RTL fake stream -> FIFO -> HLS -> DDR")

## 6. Mode 2 大 buffer 检查

如果这一格通过，说明 stream 链路相对可靠。

In [ ]:
sample_count, actual_adc_fs, actual_save_fs = configure_ctrl(
    sample_count=4096,
    adc_half_period=6,
    sample_delay=1,
    decimation=1,
    channel_mask=0b11,
    capture_mode=2
)

buf, ch0, ch1 = run_capture(sample_count, capture_mode=2)
read_status()
plot_capture(ch0, ch1, fs=actual_save_fs, title="mode 2: 4096 samples", max_points=1024)

## 7. 只打开 ADC clock

从这里开始需要接 AD9226。  
先只打开时钟，不写 DDR，建议用示波器/逻辑分析仪看 `adc_a_clk / adc_b_clk`。

In [ ]:
# 先低速，更安全：adc_half_period=25 -> 2.5MHz
sample_count, actual_adc_fs, actual_save_fs = configure_ctrl(
    sample_count=1024,
    adc_half_period=25,
    sample_delay=1,
    decimation=1,
    channel_mask=0b11,
    capture_mode=1
)

# 只 enable，不 start writer，不 start capture
ctrl.write(CTRL, 0x01)

sleep(1.0)
read_status()

print("Expected ADC clock Hz:", actual_adc_fs)

## 8. Mode 1：真实 AD9226 低速采样

先用 2.5MHz 和 1024 点。  
如果 timeout，先看 STATUS / ERROR_FLAGS / FIFO_LEVEL / SAVED_COUNTER。

In [ ]:
sample_count, actual_adc_fs, actual_save_fs = configure_ctrl(
    sample_count=1024,
    adc_half_period=25,   # 2.5 MHz 起步
    sample_delay=1,
    decimation=1,
    channel_mask=0b11,
    capture_mode=1
)

buf, ch0, ch1 = run_capture(sample_count, capture_mode=1, timeout_s=5.0)
read_status()
plot_capture(ch0, ch1, fs=actual_save_fs, title="mode 1: real AD9226 low speed")

## 9. 扫 sample_delay

真实波形不稳定时，先扫 sample_delay，不要急着改 PL。

In [ ]:
for sd in [0, 1, 2, 3]:
    print("\nTesting sample_delay =", sd)

    sample_count, actual_adc_fs, actual_save_fs = configure_ctrl(
        sample_count=1024,
        adc_half_period=25,
        sample_delay=sd,
        decimation=1,
        channel_mask=0b11,
        capture_mode=1
    )

    try:
        buf, ch0, ch1 = run_capture(sample_count, capture_mode=1, timeout_s=5.0)
        read_status()
        plot_capture(ch0, ch1, fs=actual_save_fs, title=f"real ADC sample_delay={sd}", max_points=512)
    except Exception as e:
        print("FAILED:", e)
        read_status()

## 10. 逐步提高 ADC 采样率

顺序建议：5.2MHz → 10.4MHz → 15.6MHz → 31.25MHz。  
如果某一级失败，先停在那里分析，不要继续升频。

In [ ]:
for hp in [12, 6, 4, 2]:
    print("\nTesting adc_half_period =", hp)

    sample_count, actual_adc_fs, actual_save_fs = configure_ctrl(
        sample_count=4096,
        adc_half_period=hp,
        sample_delay=1,
        decimation=1,
        channel_mask=0b11,
        capture_mode=1
    )

    try:
        buf, ch0, ch1 = run_capture(sample_count, capture_mode=1, timeout_s=5.0)
        read_status()
        plot_capture(ch0, ch1, fs=actual_save_fs, title=f"real ADC hp={hp}, Fs={actual_adc_fs/1e6:.2f} MHz", max_points=1024)
    except Exception as e:
        print("FAILED at hp =", hp, e)
        read_status()
        break

## 11. 10MHz 正弦建议测试

如果输入是 10MHz 正弦，不建议用 10.4MSPS 看波形。  
建议先用 31.25MSPS，也就是 `adc_half_period=2`。

In [ ]:
sample_count, actual_adc_fs, actual_save_fs = configure_ctrl(
    sample_count=16384,
    adc_half_period=2,   # 31.25 MSPS
    sample_delay=1,
    decimation=1,
    channel_mask=0b11,
    capture_mode=1
)

buf, ch0, ch1 = run_capture(sample_count, capture_mode=1, timeout_s=5.0)
read_status()
plot_capture(ch0, ch1, fs=actual_save_fs, title="10 MHz sine test, Fs=31.25MSPS", max_points=2000)

## 12. 62.5MSPS 最后再测

`adc_half_period=1` 是最高速，最后再测。  
如果 FIFO overflow 或 timeout，先不要怀疑 ADC，可能是 HLS writer/DDR 吞吐或外部连线问题。

In [ ]:
sample_count, actual_adc_fs, actual_save_fs = configure_ctrl(
    sample_count=32768,
    adc_half_period=1,   # 62.5 MSPS
    sample_delay=0,
    decimation=1,
    channel_mask=0b11,
    capture_mode=1
)

buf, ch0, ch1 = run_capture(sample_count, capture_mode=1, timeout_s=5.0)
read_status()
plot_capture(ch0, ch1, fs=actual_save_fs, title="real ADC, Fs=62.5MSPS", max_points=2000)

## 13. 保存 CSV

运行完任意一次 capture 后，可以保存当前 `ch0/ch1`。

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "index": np.arange(len(ch0)),
    "ch0_raw": ch0,
    "ch1_raw": ch1,
})

df.to_csv("ad9226_capture_data.csv", index=False)
df.head()